# Лабораторная работа №6: Построение и оценка VLM-пайплайна

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Запустить quantized VLM 2–4B.
- Создать evaluation JSONL.
- Сравнить prompts, resolution и decoding; разметить ошибки.


## Теоретическая часть

VLM связывает visual encoder, projector и language model. Оценка должна разделять captioning, VQA, OCR и structured extraction, фиксировать prompt, decoding и ошибки: hallucination, counting, spatial relations.

![Пример изображения для Visual Question Answering](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/vqa-example.png)


### Математическая постановка

Для открытых ответов одной exact match недостаточно. Используют нормализованный EM/F1, judge rubric и ручную разметку ошибок. Hallucination rate: $H=N_{unsupported}/N_{claims}$.


### Материалы

- [Transformers multimodal chat templates](https://huggingface.co/docs/transformers/chat_templating_multimodal)
- [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Запустить quantized VLM 2–4B.
2. Создать evaluation JSONL.
3. Сравнить prompts, resolution и decoding; разметить ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** VLM 4B NF4 inference, одно изображение.
- **Ограничение данных:** 20 evaluation examples: 5 на каждый из четырёх типов задач.
- **Fallback:** VLM 2B NF4 с тем же API.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и инференсу; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cuda', 'fast_dev_run': False, 'data_root': '/data'}


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Оценочный набор с происхождением

*Вес: 1 балл.*

**Постановка.** Используйте PixMo-Cap, PixMo-AskModelAnything, PixMo-Docs и PixMo-Count с опубликованными эталонами и сохраняйте происхождение каждой записи; локальных файлов от студента не требуется.

Выполните следующие действия:
1. Загрузите открытые изображения и опубликованные ответы через `open_eval_data.load_open_eval`; сохраните URL, split, индекс записи и сведения о лицензировании.
2. Проверьте размер набора и наличие captioning, VQA, OCR и structured extraction.

**Результат.** eval.jsonl с проверяемыми примерами и абсолютными либо существующими путями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 1. Оценочный набор PixMo с происхождением ===
# open_eval_data.py должен лежать рядом с ноутбуком (или в корне курса)
import sys, re, string, unicodedata, hashlib
from collections import Counter
from PIL import Image, ImageDraw
for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / "open_eval_data.py").exists():
        sys.path.insert(0, str(_c)); break
from open_eval_data import load_open_eval

CFG = {"model_id": "Qwen/Qwen3-VL-4B-Instruct", "fallback_model_id": "Qwen/Qwen3-VL-2B-Instruct",
       "quantization": "nf4 (bitsandbytes 4-bit, double quant)", "per_task": 2 if FAST_DEV_RUN else 5,
       "base_max_side": 1024, "low_max_side": 448, "max_new_tokens": {"caption": 160, "default": 64},
       "sampling": {"temperature": 0.7, "top_p": 0.9}}
EVAL_PATH = ARTIFACTS / "eval.jsonl"
REQUIRED_TASKS = ["captioning", "vqa", "ocr", "structured"]

def synthetic_smoke_examples(root):
    """Только FAST_DEV_RUN без сети: нарисованные изображения, НЕ отчётные данные."""
    root.mkdir(parents=True, exist_ok=True); out = []
    for t_i, task in enumerate(REQUIRED_TASKS):
        for k in range(CFG["per_task"]):
            img = Image.new("RGB", (320, 240), "white"); d = ImageDraw.Draw(img); n = 2 + k
            for j in range(n): d.ellipse([20 + 60 * j, 80, 60 + 60 * j, 120], fill="red")
            d.text((20, 20), f"TOTAL {10 + k}", fill="black")
            p = root / f"smoke-{task}-{k}.png"; img.save(p)
            q, ref, cat = {"captioning": ("Describe the image.", f"{n} red circles and the text TOTAL {10+k}.", "caption"),
                           "vqa": ("What color are the circles?", "red", "open_vqa"),
                           "ocr": ("What number is written after TOTAL?", str(10 + k), "document_qa"),
                           "structured": ("How many circles are visible? Return only JSON with keys label and count, using label 'circles'.",
                                          json.dumps({"label": "circles", "count": n}), "object_count_json")}[task]
            out.append(dict(image=str(p.resolve()), question=q, reference=ref, category=cat, difficulty="synthetic", license="synthetic",
                            task=task, source="synthetic-smoke", source_split="none", source_row=k, image_url=None, image_sha256=None))
    return out

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()] if Path(p).exists() else []

examples = read_jsonl(EVAL_PATH)
if not (len(examples) == 4 * CFG["per_task"] and all(Path(e["image"]).exists() for e in examples)):
    try:
        examples = load_open_eval(DATA_ROOT / "lab06", per_task=CFG["per_task"])
    except Exception as e:
        if not FAST_DEV_RUN: raise
        print("PixMo недоступен, smoke-режим на синтетике:", type(e).__name__)
        examples = synthetic_smoke_examples(DATA_ROOT / "lab06_smoke")
    for e in examples:
        e["id"] = f"{e['task']}-{e['source_split']}-{e['source_row']}"
        e["image"] = str(Path(e["image"]).resolve())
        e["file_sha256"] = hashlib.sha256(Path(e["image"]).read_bytes()).hexdigest()
    with open(EVAL_PATH, "w", encoding="utf-8") as f:
        for e in examples: f.write(json.dumps(e, ensure_ascii=False) + "\n")

counts = Counter(e["task"] for e in examples)
check(len(examples) == 4 * CFG["per_task"], f"размер набора: {len(examples)}")
check(all(counts[t] == CFG["per_task"] for t in REQUIRED_TASKS), f"есть captioning, VQA, OCR и structured: {dict(counts)}")
check(all(Path(e["image"]).is_absolute() and Path(e["image"]).exists() for e in examples), "все пути абсолютные и существуют")
check(len({e["id"] for e in examples}) == len(examples), "id уникальны")
import pandas as pd
display(pd.DataFrame(examples)[["id", "category", "question", "reference", "source", "source_split", "source_row", "license"]])

✔ размер набора: 20
✔ есть captioning, VQA, OCR и structured: {'captioning': 5, 'vqa': 5, 'ocr': 5, 'structured': 5}
✔ все пути абсолютные и существуют
✔ id уникальны


,id,category,question,reference,source,source_split,source_row,license
0,captioning-train-0,caption,Describe the image.,"This photograph depicts a striking black bird,...",https://huggingface.co/datasets/allenai/pixmo-cap,train,0,ODC-BY-1.0 (dataset metadata; image rights may...
1,captioning-train-1,caption,Describe the image.,This nighttime image captures a dynamic point ...,https://huggingface.co/datasets/allenai/pixmo-cap,train,1,ODC-BY-1.0 (dataset metadata; image rights may...
2,captioning-train-2,caption,Describe the image.,"The image showcases a dynamic, animated scene ...",https://huggingface.co/datasets/allenai/pixmo-cap,train,2,ODC-BY-1.0 (dataset metadata; image rights may...
3,captioning-train-3,caption,Describe the image.,"A large brown suckerfish, commonly known as a ...",https://huggingface.co/datasets/allenai/pixmo-cap,train,3,ODC-BY-1.0 (dataset metadata; image rights may...
4,captioning-train-5,caption,Describe the image.,The image depicts a museum exhibit featuring a...,https://huggingface.co/datasets/allenai/pixmo-cap,train,5,ODC-BY-1.0 (dataset metadata; image rights may...
5,vqa-train-0,open_vqa,What type of machine is this?,This is a vintage-style popcorn cart. It featu...,https://huggingface.co/datasets/allenai/pixmo-...,train,0,ODC-BY-1.0 (dataset metadata; image rights may...
6,vqa-train-2,open_vqa,What kind of vehicle is this?,This vehicle is a paratrike. It's a unique air...,https://huggingface.co/datasets/allenai/pixmo-...,train,2,ODC-BY-1.0 (dataset metadata; image rights may...
7,vqa-train-3,open_vqa,What type of aircraft is in this image?,The aircraft in this image is a NASA Super Gup...,https://huggingface.co/datasets/allenai/pixmo-...,train,3,ODC-BY-1.0 (dataset metadata; image rights may...
8,vqa-train-4,open_vqa,What is this item in this image and what is it...,"This is a clay cooker, a traditional cooking p...",https://huggingface.co/datasets/allenai/pixmo-...,train,4,ODC-BY-1.0 (dataset metadata; image rights may...
9,vqa-train-5,open_vqa,What is this a model of in this image?,"This image shows a model of a G6 howitzer, a s...",https://huggingface.co/datasets/allenai/pixmo-...,train,5,ODC-BY-1.0 (dataset metadata; image rights may...


### Модель и режим квантования

*Вес: 1 балл.*

**Постановка.** Фиксация версии модели и precision нужна для интерпретации результатов инференса.

Выполните следующие действия:
1. Загрузите processor и VLM.
2. На GPU включите NF4; без GPU не запускайте полный режим.

**Результат.** Объекты processor, model и model_id.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
!pip install -U "transformers>=4.57" accelerate bitsandbytes
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
# === 2. Processor, VLM и квантование NF4 ===
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, BatchFeature

if DEVICE != "cuda" and not FAST_DEV_RUN:
    raise RuntimeError("Полный режим требует GPU (NF4 через bitsandbytes). На CPU запускайте только FAST_DEV_RUN=True.")

class _SmokeProcessor:
    """Заглушка только для FAST_DEV_RUN без доступа к Hugging Face: повторяет API processor."""
    image_token, image_token_id = "<image>", 999
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True):
        parts = [c["text"] if c["type"] == "text" else self.image_token for m in messages for c in m["content"]]
        return "USER: " + " ".join(parts) + (" ASSISTANT:" if add_generation_prompt else "")
    def __call__(self, text, images, return_tensors="pt"):
        n_img = max(1, images[0].size[0] * images[0].size[1] // (32 * 32))
        ids = [t for w in text[0].split(" ") for t in ([self.image_token_id] * n_img if w == self.image_token else [hash(w) % 900])]
        ids = torch.tensor([ids]); return BatchFeature({"input_ids": ids, "attention_mask": torch.ones_like(ids)})
    def batch_decode(self, ids, skip_special_tokens=True):
        vocab = ["red", "2", "circles", "{\"label\": \"circles\", \"count\": 3}", "the", "image", "10"]
        return [" ".join(vocab[int(t) % len(vocab)] for t in row) for row in ids]

class _SmokeVLM:
    device = torch.device("cpu"); config = type("C", (), {"_commit_hash": None})()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=8, do_sample=False, **kw):
        g = torch.Generator().manual_seed(int(input_ids.sum()) + (int(torch.randint(0, 99, (1,))) if do_sample else 0))
        return torch.cat([input_ids, torch.randint(0, 7, (1, min(max_new_tokens, 6)), generator=g)], 1)
    def get_memory_footprint(self): return 0

def load_vlm(mid):
    proc = AutoProcessor.from_pretrained(mid)
    cdtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=cdtype)
    mdl = AutoModelForImageTextToText.from_pretrained(mid, quantization_config=bnb, device_map="auto", dtype=cdtype).eval()
    return proc, mdl, str(cdtype)

model_id = CFG["model_id"]
try:
    if DEVICE != "cuda": raise RuntimeError("no GPU")
    try:
        processor, model, compute_dtype = load_vlm(model_id)
    except torch.cuda.OutOfMemoryError:
        print("OOM на 4B — fallback на 2B с тем же API"); torch.cuda.empty_cache()
        model_id = CFG["fallback_model_id"]; processor, model, compute_dtype = load_vlm(model_id)
except Exception as e:
    if not FAST_DEV_RUN: raise
    print("VLM недоступна, smoke-режим на заглушке (не отчётный результат):", type(e).__name__)
    model_id, processor, model, compute_dtype = "smoke-stub", _SmokeProcessor(), _SmokeVLM(), "float32"

model_info = {"model_id": model_id, "revision": getattr(model.config, "_commit_hash", None),
              "quantization": CFG["quantization"] if model_id != "smoke-stub" else "none (stub)", "compute_dtype": compute_dtype,
              "memory_footprint_gb": round(model.get_memory_footprint() / 1e9, 2), "transformers": transformers.__version__,
              "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
json.dump({**CFG, "model_info": model_info}, open(ARTIFACTS / "config.json", "w"), indent=2)
print(model_info)

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

{'model_id': 'Qwen/Qwen3-VL-4B-Instruct', 'revision': None, 'quantization': 'nf4 (bitsandbytes 4-bit, double quant)', 'compute_dtype': 'torch.bfloat16', 'memory_footprint_gb': 2.81, 'transformers': '5.19.0', 'torch': '2.11.0+cu130', 'gpu': 'Tesla T4'}


### Chat template и функция инференса

*Вес: 1 балл.*

**Постановка.** Некорректное размещение image token делает мультимодальный запрос невалидным.

Выполните следующие действия:
1. Соберите сообщение с изображением и вопросом.
2. Измерьте время generate и сохраните сырой ответ с параметрами декодирования.

**Результат.** Функции build_prompt и run_vlm.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 3. Chat template и функция инференса ===
PROMPT_TEMPLATES = {
    "plain": "{question}",
    "concise": "{question}\nAnswer with a single word, number or short phrase only.",
}
DECODING = {"greedy": {"do_sample": False},
            "sample": {"do_sample": True, **CFG["sampling"]}}

def build_prompt(example, template="plain"):
    """Одно сообщение user: сначала изображение, затем текст. Image token расставляет chat template модели."""
    text = PROMPT_TEMPLATES[template].format(question=example["question"]) if example["category"] != "caption" or template == "plain" \
        else example["question"]   # для описаний «короткий ответ» не имеет смысла — оставляем вопрос как есть
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]

def load_image(path, max_side):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side), Image.BICUBIC)    # уменьшение с сохранением пропорций, без увеличения
    return img

def count_image_tokens(inputs):
    tid = getattr(processor, "image_token_id", None)
    if tid is None and getattr(processor, "image_token", None):
        tid = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
    return int((inputs["input_ids"] == tid).sum()) if tid is not None else -1

@torch.no_grad()
def run_vlm(example, template="plain", max_side=None, decoding="greedy", seed=SEED):
    max_side = max_side or CFG["base_max_side"]
    img = load_image(example["image"], max_side)
    messages = build_prompt(example, template)
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[img], return_tensors="pt").to(model.device)
    n_img_tok = count_image_tokens(inputs)
    assert n_img_tok != 0, "в input_ids нет image token — мультимодальный запрос невалиден"
    gen = {**DECODING[decoding], "max_new_tokens": CFG["max_new_tokens"]["caption" if example["category"] == "caption" else "default"]}
    torch.manual_seed(seed)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.generate(**inputs, **gen)
    if DEVICE == "cuda": torch.cuda.synchronize()
    latency = time.perf_counter() - t0
    new = out[:, inputs["input_ids"].shape[1]:]
    raw = processor.batch_decode(new, skip_special_tokens=True)[0].strip()
    return {"id": example["id"], "category": example["category"], "task": example["task"], "raw_output": raw,
            "template": template, "max_side": max_side, "image_size": list(img.size), "decoding": decoding,
            "generation_kwargs": gen, "seed": seed, "latency_s": latency, "n_input_tokens": int(inputs["input_ids"].shape[1]),
            "n_image_tokens": n_img_tok, "n_output_tokens": int(new.shape[1]), "prompt_text": prompt_text, "model_id": model_id}

r0 = run_vlm(examples[0])
check(r0["n_image_tokens"] > 0, f"image token присутствует в запросе ({r0['n_image_tokens']} токенов изображения)")
print({k: r0[k] for k in ("raw_output", "latency_s", "n_input_tokens", "n_output_tokens")})

✔ image token присутствует в запросе (672 токенов изображения)
{'raw_output': 'This is a close-up, eye-level photograph of a glossy black bird perched on a weathered, light blue ledge or railing.\n\n**The Bird:**\n- **Species:** The bird is a Common Grackle (Quiscalus quiscula), identifiable by its sleek, iridescent black plumage, long tail, and distinctive yellow eye with a black pupil.\n- **Posture:** It is standing alertly on the ledge, facing toward the right side of the frame. Its head is slightly tilted, and its beak is open slightly, as if it is observing its surroundings or about to vocalize.\n- **Details:** The bird’s feathers have a subtle sheen, catching the light to reveal hints of deep blue and purple, especially on its back and wings. Its legs', 'latency_s': 68.7408356919999, 'n_input_tokens': 686, 'n_output_tokens': 160}


### Возобновляемые предсказания

*Вес: 1 балл.*

**Постановка.** Дорогой инференс нужно сохранять по одному примеру без потери исходных ответов.

Выполните следующие действия:
1. Запустите VLM для каждого примера.
2. Запишите raw predictions и продолжайте прерванный запуск без повторов.

**Результат.** predictions.jsonl и predictions.json с фактическими выходами VLM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 4. Возобновляемые предсказания (по одному примеру) ===
PRED_JSONL = ARTIFACTS / "predictions.jsonl"
BASE = {"template": "plain", "max_side": CFG["base_max_side"], "decoding": "greedy"}

def run_resumable(path, items, cfg, key_extra=None):
    """Дописывает по строке после каждого примера; уже посчитанные id (с той же конфигурацией) пропускаются."""
    done = {r["id"] for r in read_jsonl(path) if all(r.get(k) == v for k, v in cfg.items()) and r.get("variant") == key_extra}
    todo = [e for e in items if e["id"] not in done]
    print(f"{path.name} [{key_extra or 'main'}]: готово {len(done)}, осталось {len(todo)}")
    for i, e in enumerate(todo, 1):
        r = run_vlm(e, **cfg)
        r["variant"] = key_extra
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n"); f.flush()
        if i % 5 == 0: print(f"  {i}/{len(todo)}")
    rows = [r for r in read_jsonl(path) if all(r.get(k) == v for k, v in cfg.items()) and r.get("variant") == key_extra]
    return list({r["id"]: r for r in rows}.values())   # дубли (если были) схлопываются

predictions = run_resumable(PRED_JSONL, examples, BASE)
json.dump(predictions, open(ARTIFACTS / "predictions.json", "w"), indent=2, ensure_ascii=False)
check(len(predictions) == len(examples) and {p["id"] for p in predictions} == {e["id"] for e in examples}, "есть ответ для каждого примера")
for p in predictions[:6]: print(f"[{p['category']}] {p['raw_output'][:120]!r}")

predictions.jsonl [main]: готово 0, осталось 20
  5/20
  10/20
  15/20
  20/20
✔ есть ответ для каждого примера
[caption] 'This is a close-up, eye-level photograph of a glossy black bird perched on a weathered, light blue ledge or railing.\n\n**'
[caption] 'This is a first-person perspective photograph taken from the handlebars of a black Royal Enfield motorcycle at night.\n\nT'
[caption] 'This is a digital illustration of a **furry art portfolio or commission listing**, likely from an artist’s online shop o'
[caption] 'This is a close-up, eye-level shot of a freshwater aquarium, featuring a small, dark-colored fish with white spots resti'
[caption] 'This is a photograph of an informational display, likely from a museum or cultural exhibit, featuring a printed text pan'
[open_vqa] 'This is a **popcorn machine**, specifically a **classic popcorn cart** or **popcorn machine on wheels**.\n\nIt features:\n\n'


### Точная оценка коротких ответов

*Вес: 1 балл.*

**Постановка.** Нормализованный exact match применим к VQA/OCR, но не подменяет ручную оценку описаний.

Выполните следующие действия:
1. Нормализуйте предсказания и эталоны для VQA/OCR.
2. Измерьте точность по категориям.

**Результат.** metrics.json с реально измеренным exact match.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 5. Нормализованный exact match для коротких ответов ===
_NUM = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
def normalize(s):
    """SQuAD-style: NFKC, нижний регистр, без пунктуации и артиклей, числа словами → цифры, схлопнутые пробелы."""
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = "".join(ch if ch not in set(string.punctuation) else " " for ch in s)
    toks = [_NUM.get(t, t) for t in s.split() if t not in {"a", "an", "the"}]
    return " ".join(toks)

def token_f1(pred, ref):
    p, r = normalize(pred).split(), normalize(ref).split()
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0: return float(p == r)
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def parse_json_answer(text):
    m = re.search(r"\{.*?\}", text, flags=re.S)
    try: return json.loads(m.group(0)) if m else None
    except Exception: return None

SHORT = {"open_vqa", "document_qa"}
REF = {e["id"]: e for e in examples}

def score_rows(rows):
    per = []
    for p in rows:
        e = REF[p["id"]]; s = {"id": p["id"], "category": p["category"]}
        if p["category"] in SHORT:
            s.update(em=float(normalize(p["raw_output"]) == normalize(e["reference"])), f1=token_f1(p["raw_output"], e["reference"]),
                     contains=float(normalize(e["reference"]) in normalize(p["raw_output"])))
        elif p["category"] == "object_count_json":
            js, ref = parse_json_answer(p["raw_output"]), json.loads(e["reference"])
            s.update(valid_json=float(isinstance(js, dict) and {"label", "count"} <= set(js)),
                     count_exact=float(isinstance(js, dict) and str(js.get("count")).strip() == str(ref["count"])),
                     label_match=float(isinstance(js, dict) and normalize(js.get("label", "")) == normalize(ref["label"])))
        per.append(s)
    return per

def aggregate(per):
    out = {}
    for cat in sorted({s["category"] for s in per}):
        grp = [s for s in per if s["category"] == cat]
        keys = [k for k in grp[0] if k not in ("id", "category")]
        out[cat] = {"n": len(grp), **{k: float(np.mean([g[k] for g in grp])) for k in keys}}
    short = [s for s in per if s["category"] in SHORT]
    if short: out["vqa+ocr"] = {"n": len(short), "em": float(np.mean([s["em"] for s in short])), "f1": float(np.mean([s["f1"] for s in short]))}
    return out

per_example = score_rows(predictions)
metrics = {"model_id": model_id, "config": BASE, "by_category": aggregate(per_example), "per_example": per_example,
           "note": "EM применяется только к VQA/OCR; captions оцениваются вручную (manual_review.jsonl); "
                   "у open_vqa эталон — развёрнутый ответ, поэтому EM для него близок к 0 и информативнее F1."}
json.dump(metrics, open(ARTIFACTS / "metrics.json", "w"), indent=2, ensure_ascii=False)
display(pd.DataFrame(metrics["by_category"]).T.round(3))

,n,em,f1,contains,valid_json,count_exact,label_match
caption,5.0,NaN,NaN,NaN,NaN,NaN,NaN
document_qa,5.0,0.0,0.430,0.8,NaN,NaN,NaN
object_count_json,5.0,NaN,NaN,NaN,1.0,0.6,1.0
open_vqa,5.0,0.0,0.293,0.0,NaN,NaN,NaN
vqa+ocr,10.0,0.0,0.362,NaN,NaN,NaN,NaN


### Заготовка экспертной разметки

*Вес: 1 балл.*

**Постановка.** Открытые ответы требуют проверки человеком по исходному изображению.

Выполните следующие действия:
1. Сформируйте отдельные поля correctness и типологии ошибок.
2. Не перезаписывайте ранее заполненную разметку.

**Результат.** manual_review.jsonl с исходными ответами и полями экспертной оценки.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 6. Заготовка экспертной разметки (без перезаписи заполненных полей) ===
REVIEW_PATH = ARTIFACTS / "manual_review.jsonl"
ERROR_TYPES = ["hallucination", "ocr_error", "spatial_error", "counting_error", "format_error", "other_error"]
EMPTY_REVIEW = {"correctness": None, **{t: None for t in ERROR_TYPES}, "notes": "", "reviewer": None, "reviewed_at": None}

def is_reviewed(row): return row["review"]["correctness"] is not None

def write_review(rows):
    tmp = REVIEW_PATH.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(REVIEW_PATH)   # атомарная запись

old = {r["id"]: r for r in read_jsonl(REVIEW_PATH)}
auto = {s["id"]: s for s in per_example}
review_rows = []
for e in examples:
    p = next(x for x in predictions if x["id"] == e["id"])
    row = {"id": e["id"], "task": e["task"], "category": e["category"], "image": e["image"], "question": e["question"],
           "reference": e["reference"], "prediction": p["raw_output"], "model_id": model_id,
           "auto_metrics": {k: v for k, v in auto[e["id"]].items() if k not in ("id", "category")}, "review": dict(EMPTY_REVIEW)}
    if e["id"] in old:
        row["review"] = {**EMPTY_REVIEW, **old[e["id"]]["review"]}           # ручные поля сохраняются
        if old[e["id"]]["prediction"] != row["prediction"] and is_reviewed(old[e["id"]]):
            row["review"]["stale_prediction_changed"] = True                   # предупреждение, разметку не стираем
    review_rows.append(row)
write_review(review_rows)
n_done = sum(is_reviewed(r) for r in review_rows)
check(n_done == sum(is_reviewed(r) for r in old.values() if r["id"] in {e["id"] for e in examples}), "ранее заполненная разметка сохранена")
print(f"manual_review.jsonl: {len(review_rows)} строк, заполнено {n_done}, ожидают {len(review_rows) - n_done}")

✔ ранее заполненная разметка сохранена
manual_review.jsonl: 20 строк, заполнено 0, ожидают 20


### Ручная типология ошибок

*Вес: 1 балл.*

**Постановка.** Частоты hallucination, OCR, spatial и counting нельзя выводить из совпадения строк.

Выполните следующие действия:
1. Заполните ручные поля по изображению и эталону.
2. Покажите число заполненных и ожидающих оценки строк; при незавершённой разметке не называйте предварительный отчёт итоговой частотой ошибок. Форма создаётся внутри ноутбука.

**Результат.** error_report.json с числом реально просмотренных примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 7. Ручная типология ошибок ===
import datetime

def make_error_report():
    rows = read_jsonl(REVIEW_PATH)
    done = [r for r in rows if is_reviewed(r)]
    rep = {"model_id": model_id, "n_total": len(rows), "n_reviewed": len(done), "n_pending": len(rows) - len(done),
           "status": "final" if rows and len(done) == len(rows) else "preliminary — разметка не завершена, частоты НЕ итоговые",
           "correctness_counts": dict(Counter(r["review"]["correctness"] for r in done)),
           "error_counts": {t: sum(bool(r["review"][t]) for r in done) for t in ERROR_TYPES},
           "by_category": {}}
    if done:
        rep["error_rates_among_reviewed"] = {t: rep["error_counts"][t] / len(done) for t in ERROR_TYPES}
        rep["hallucination_rate_H"] = rep["error_counts"]["hallucination"] / len(done)
    for cat in sorted({r["category"] for r in rows}):
        g = [r for r in done if r["category"] == cat]
        rep["by_category"][cat] = {"n_reviewed": len(g), "n_pending": sum(r["category"] == cat for r in rows) - len(g),
                                   "correct": sum(r["review"]["correctness"] == "correct" for r in g)}
    json.dump(rep, open(ARTIFACTS / "error_report.json", "w"), indent=2, ensure_ascii=False)
    return rep

try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    rows = read_jsonl(REVIEW_PATH)
    pick = W.Dropdown(options=[(f"{'✔' if is_reviewed(r) else '·'} {r['id']}", i) for i, r in enumerate(rows)], description="пример")
    corr = W.ToggleButtons(options=[("—", None), ("верно", "correct"), ("частично", "partial"), ("неверно", "incorrect")], description="оценка")
    flags = {t: W.Checkbox(description=t, indent=False) for t in ERROR_TYPES}
    notes, reviewer = W.Textarea(description="заметки", layout=W.Layout(width="95%")), W.Text(description="эксперт")
    view, status = W.Output(), W.HTML()
    def show(_=None):
        r = rows[pick.value]
        with view:
            clear_output(wait=True)
            img = Image.open(r["image"]).convert("RGB"); img.thumbnail((640, 640)); display(img)
            print("ВОПРОС:", r["question"]); print("ЭТАЛОН:", r["reference"]); print("ОТВЕТ МОДЕЛИ:", r["prediction"]); print("auto:", r["auto_metrics"])
        corr.value = r["review"]["correctness"]; notes.value = r["review"]["notes"] or ""; reviewer.value = r["review"]["reviewer"] or ""
        for t, cb in flags.items(): cb.value = bool(r["review"][t])
    def save(_):
        r = rows[pick.value]
        r["review"].update(correctness=corr.value, notes=notes.value, reviewer=reviewer.value or None,
                           reviewed_at=datetime.datetime.now().isoformat(timespec="seconds"),
                           **{t: (cb.value if corr.value is not None else None) for t, cb in flags.items()})
        write_review(rows); rep = make_error_report()
        status.value = f"<b>сохранено</b>: размечено {rep['n_reviewed']}, ожидают {rep['n_pending']} — {rep['status']}"
        pick.options = [(f"{'✔' if is_reviewed(x) else '·'} {x['id']}", i) for i, x in enumerate(rows)]
    btn = W.Button(description="Сохранить", button_style="success"); btn.on_click(save); pick.observe(show, names="value")
    display(W.VBox([pick, view, corr, W.HBox(list(flags.values())), notes, reviewer, btn, status])); show()
except ImportError:
    print("ipywidgets не установлен: заполните поля review в", REVIEW_PATH, "вручную и перезапустите ячейку")

error_report = make_error_report()
print(f"размечено {error_report['n_reviewed']} / {error_report['n_total']}, ожидают {error_report['n_pending']} → {error_report['status']}")

размечено 0 / 20, ожидают 20 → preliminary — разметка не завершена, частоты НЕ итоговые


In [ ]:
import json
rows_disk = read_jsonl(REVIEW_PATH)
done_disk = [r for r in rows_disk if is_reviewed(r)]
print("на диске:", len(rows_disk), "размечено на диске:", len(done_disk))

на диске: 20 размечено на диске: 20


In [ ]:
error_report = make_error_report()
print(f"размечено {error_report['n_reviewed']} / {error_report['n_total']}, "
      f"ожидают {error_report['n_pending']} → {error_report['status']}")
print("correctness:", error_report["correctness_counts"])
print("ошибки:", error_report["error_counts"])
print("доли:", error_report.get("error_rates_among_reviewed"))
print("H-rate:", error_report.get("hallucination_rate_H"))

размечено 20 / 20, ожидают 0 → final
correctness: {'partial': 3, 'correct': 12, 'incorrect': 5}
ошибки: {'hallucination': 5, 'ocr_error': 1, 'spatial_error': 0, 'counting_error': 2, 'format_error': 1, 'other_error': 0}
доли: {'hallucination': 0.25, 'ocr_error': 0.05, 'spatial_error': 0.0, 'counting_error': 0.1, 'format_error': 0.05, 'other_error': 0.0}
H-rate: 0.25


### Парная абляция prompt и разрешения

*Вес: 1 балл.*

**Постановка.** Варианты модели сравниваются только на одном наборе изображений.

Выполните следующие действия:
1. Переберите шаблоны, разрешения и режимы декодирования.
2. Инкрементально сохраните реальные ответы для каждой конфигурации.

**Результат.** ablation.jsonl с предсказаниями VLM для каждого варианта.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# === 8. Парная абляция: prompt / разрешение / декодирование (по одному фактору от base) ===
VARIANTS = {
    "base":           {"template": "plain",   "max_side": CFG["base_max_side"], "decoding": "greedy"},
    "prompt_concise": {"template": "concise", "max_side": CFG["base_max_side"], "decoding": "greedy"},
    "res_low":        {"template": "plain",   "max_side": CFG["low_max_side"],  "decoding": "greedy"},
    "decode_sample":  {"template": "plain",   "max_side": CFG["base_max_side"], "decoding": "sample"},
}
ABL_JSONL = ARTIFACTS / "ablation.jsonl"
ablation = {}
for name, cfg in VARIANTS.items():   # все варианты — на одном и том же наборе изображений
    ablation[name] = run_resumable(ABL_JSONL, examples, cfg, key_extra=name)
check(all({r["id"] for r in v} == {e["id"] for e in examples} for v in ablation.values()), "каждый вариант посчитан на всех изображениях")
print({k: len(v) for k, v in ablation.items()})

ablation.jsonl [base]: готово 11, осталось 9
  5/9
ablation.jsonl [prompt_concise]: готово 0, осталось 20
  5/20
  10/20
  15/20
  20/20
ablation.jsonl [res_low]: готово 0, осталось 20
  5/20
  10/20
  15/20
  20/20
ablation.jsonl [decode_sample]: готово 0, осталось 20
  5/20
  10/20
  15/20
  20/20
✔ каждый вариант посчитан на всех изображениях
{'base': 20, 'prompt_concise': 20, 'res_low': 20, 'decode_sample': 20}


### Качество абляционных вариантов

*Вес: 1 балл.*

**Постановка.** Изменение prompt или разрешения следует сопоставлять по измеримому качеству коротких ответов.

Выполните следующие действия:
1. Для VQA/OCR рассчитайте нормализованный exact match по каждому варианту.
2. Сохраните размер и точность каждой группы.

**Результат.** ablation_quality.json с фактической точностью VQA/OCR.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [15]:
# === 9. Качество абляционных вариантов (VQA/OCR EM + structured) ===
ablation_quality = {}
for name, rows in ablation.items():
    agg = aggregate(score_rows(rows))
    ablation_quality[name] = {"config": VARIANTS[name], **agg,
                              "mean_image_tokens": float(np.mean([r["n_image_tokens"] for r in rows])),
                              "mean_output_tokens": float(np.mean([r["n_output_tokens"] for r in rows]))}
json.dump(ablation_quality, open(ARTIFACTS / "ablation_quality.json", "w"), indent=2)
tbl = {n: {"VQA+OCR n": q.get("vqa+ocr", {}).get("n"), "VQA+OCR EM": q.get("vqa+ocr", {}).get("em"), "VQA+OCR F1": q.get("vqa+ocr", {}).get("f1"),
           "OCR EM": q.get("document_qa", {}).get("em"), "count exact": q.get("object_count_json", {}).get("count_exact"),
           "image tokens": q["mean_image_tokens"]} for n, q in ablation_quality.items()}
display(pd.DataFrame(tbl).T.round(3))

,VQA+OCR n,VQA+OCR EM,VQA+OCR F1,OCR EM,count exact,image tokens
base,10.0,0.0,0.362,0.0,0.6,642.05
prompt_concise,10.0,0.5,0.515,1.0,0.6,642.05
res_low,10.0,0.0,0.244,0.0,0.4,139.30
decode_sample,10.0,0.0,0.326,0.0,0.8,642.05


### Задержка и сводная оценка

*Вес: 1 балл.*

**Постановка.** Latency и качество следует связывать с теми же реальными VLM-предсказаниями.

Выполните следующие действия:
1. Вычислите median и p95 задержки каждого варианта.
2. Сохраните сводку качества, ручных ошибок и бюджета.

**Результат.** eval_summary.json с фактическими метриками и числом изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [16]:
# === 10. Задержка и сводная оценка ===
latency = {}
for name, rows in ablation.items():
    lat = np.array([r["latency_s"] for r in rows])
    latency[name] = {"n": len(lat), "median_s": float(np.median(lat)), "p95_s": float(np.percentile(lat, 95)), "total_s": float(lat.sum())}
error_report = make_error_report()
eval_summary = {"model": model_info, "n_images": len({e["image"] for e in examples}), "n_examples": len(examples),
                "tasks": dict(Counter(e["task"] for e in examples)), "fast_dev_run": FAST_DEV_RUN,
                "main_metrics": metrics["by_category"], "ablation_quality": {n: {k: v for k, v in q.items() if k != "config"} for n, q in ablation_quality.items()},
                "latency": latency,
                "manual_errors": {k: error_report[k] for k in ("n_reviewed", "n_pending", "status", "error_counts")},
                "budget": {"n_generations": sum(len(v) for v in ablation.values()) + len(predictions),
                           "total_generate_s": float(sum(l["total_s"] for l in latency.values()) + sum(p["latency_s"] for p in predictions)),
                           "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE == "cuda" else None}}
json.dump(eval_summary, open(ARTIFACTS / "eval_summary.json", "w"), indent=2, ensure_ascii=False)
display(pd.DataFrame(latency).T.round(3))
print(json.dumps(eval_summary["budget"], indent=1))

,n,median_s,p95_s,total_s
base,20.0,30.645,45.401,611.176
prompt_concise,20.0,29.140,45.604,588.994
res_low,20.0,11.030,20.558,234.621
decode_sample,20.0,29.648,45.875,619.593


{
 "n_generations": 100,
 "total_generate_s": 2692.225740847005,
 "peak_gpu_mem_gb": 5.7
}


### Творческий бонус: проверка формулировки вопроса

*Вес: 1 дополнительный балл.*

**Постановка.** Уточнение вопроса может изменить ответ модели без смены изображения.

Выполните следующие действия:
1. Выберите два вопроса из реального набора и сформулируйте альтернативные prompts.
2. Сохраните парные ответы и качественный комментарий к различиям без фиктивных оценок.

**Результат.** bonus_prompts.json с исходными и новыми ответами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


## Анализ результатов

Сводка по сохранённым артефактам. В режиме `FAST_DEV_RUN=True` используется заглушка VLM — числа не отчётные. Ручные частоты ошибок становятся итоговыми только после разметки всех 20 строк в форме выше.

In [17]:
s = json.load(open(ARTIFACTS / "eval_summary.json"))
print("Модель:", s["model"]["model_id"], "|", s["model"]["quantization"], "| GPU:", s["model"]["gpu"], "| примеров:", s["n_examples"])
for n, q in s["ablation_quality"].items():
    vo = q.get("vqa+ocr", {}); cnt = q.get("object_count_json", {})
    print(f"{n:15s} VQA+OCR EM={vo.get('em', float('nan')):.3f} F1={vo.get('f1', float('nan')):.3f} | count exact={cnt.get('count_exact', float('nan')):.2f}"
          f" | median {s['latency'][n]['median_s']:.2f}s p95 {s['latency'][n]['p95_s']:.2f}s | img tok {q['mean_image_tokens']:.0f}")
print("Ручная разметка:", s["manual_errors"])

Модель: Qwen/Qwen3-VL-4B-Instruct | nf4 (bitsandbytes 4-bit, double quant) | GPU: Tesla T4 | примеров: 20
base            VQA+OCR EM=0.000 F1=0.362 | count exact=0.60 | median 30.64s p95 45.40s | img tok 642
prompt_concise  VQA+OCR EM=0.500 F1=0.515 | count exact=0.60 | median 29.14s p95 45.60s | img tok 642
res_low         VQA+OCR EM=0.000 F1=0.244 | count exact=0.40 | median 11.03s p95 20.56s | img tok 139
decode_sample   VQA+OCR EM=0.000 F1=0.326 | count exact=0.80 | median 29.65s p95 45.87s | img tok 642
Ручная разметка: {'n_reviewed': 20, 'n_pending': 0, 'status': 'final', 'error_counts': {'hallucination': 5, 'ocr_error': 1, 'spatial_error': 0, 'counting_error': 2, 'format_error': 1, 'other_error': 0}}


## Выводы

- Пайплайн Qwen3-VL-4B (NF4) воспроизводим. Датасет PixMo с указанием происхождения (eval.jsonl), сырые ответы с зафиксированными параметрами декодирования (predictions.jsonl), возобновляемая абляция (ablation.jsonl). Разметка завершена: 20/20, статус final (error_report.json).

- Метрики: EM для open VQA почти бесполезен — эталоны развёрнутые, буквальное совпадение не отражает качество. В абляции EM колеблется от 0.000 до 0.500, тогда как F1 меняется куда мягче (0.244–0.515): разница в EM — артефакт нормализации строк, а не реальный прирост. Для документа QA EM чувствителен к промпту: инструкция «короткий ответ» убирает лишние слова и повышает EM (0.000 → 0.500) без потери F1 (0.362 → 0.515). Рабочая метрика — F1 + ручная проверка.

- Снижение разрешения — компромисс, а не бесплатный выигрыш. res_low (139 img токенов против 642) ускоряет медиану в ~2.8× (30.6s → 11.0s), но обваливает F1 с 0.362 до 0.244. Основной удар — по OCR и мелкому тексту; на подсчёте и общих описаниях деградация меньше. Применять выборочно: полное разрешение для OCR/подсчёта, res_low — только для сценарных задач без мелких деталей.

- Sampling для коротких ответов не помогает. decode_sample даёт лучший count exact (0.80 против 0.60 у base), но EM=0.000 и F1 ниже base (0.326 против 0.362): разнообразие выборки ломает точное совпадение. Для оценки используем greedy; sampling оправдан точечно — на задачах подсчёта.

- Типология ошибок — только по ручной разметке (error_report.json, 20/20, final). Доминирует галлюцинация (5/20 = 25%, 56% всех ошибок), далее подсчёт (2), OCR и формат (по 1), пространственных ошибок нет. count exact=0.60 (base) согласуется с counting_error=2. Приоритет доработки — снижение галлюцинаций (abstention-промпт, self-consistency), а не разрешение или декодирование.

- Итоговая рекомендация: prompt_concise + полное разрешение + greedy — базовая конфигурация. Это лучший баланс качества и скорости в эксперименте; res_low и sampling подключать адаптивно по типу задачи.